# Day 2 – Module 2: Working with Hugging Face Models

**Topics covered:**
- Text summarization
- Question answering
- Sentiment analysis
- Image classification

**What you will do in this notebook:**
Run four common AI tasks using the Hugging Face `pipeline` API. Each section follows the same pattern: enterprise scenario, load a pipeline with a pre-trained model, run inference on real inputs, and inspect outputs. Model names come from `config.json`; no API keys are required — all models run locally.

## 1. The Hugging Face pipeline pattern

Every task follows the same three-step pattern:

```
┌──────────────────────────────────────────────────────────────────────┐
│              HUGGING FACE PIPELINE PATTERN                           │
│                                                                      │
│  Step 1: Load              Step 2: Call              Step 3: Read    │
│                                                                      │
│  pipe = pipeline(          result = pipe(            result =        │
│    "task-name",              "input text"              [{'label':    │
│    model="model-id"          or image_url                'POSITIVE', │
│  )                         )                            'score':     │
│                                                          0.99}]      │
│                                                                      │
│  Downloads model           Tokenize + inference      Structured     │
│  from Hub (cached)         on CPU or GPU             dict output    │
└──────────────────────────────────────────────────────────────────────┘
```

| Task | Pipeline name | Input | Output |
|------|--------------|-------|--------|
| Summarization | `"summarization"` | Long text (str) | `[{"summary_text": "..."}]` |
| Question answering | `"question-answering"` | question + context (str) | `{"answer": "...", "score": float, "start": int, "end": int}` |
| Sentiment analysis | `"sentiment-analysis"` | Text (str or list) | `[{"label": "POSITIVE", "score": float}]` |
| Image classification | `"image-classification"` | Image path/URL | `[{"label": "...", "score": float}, ...]` |

## 2. Setup

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

with open(Path.cwd() / "config.json") as f:
    CONFIG = json.load(f)

print("Model configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

import torch
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1
device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device_name}")

## 3. Text summarization

**Scenario:** An internal reporting tool receives long financial or operational reports. The tool auto-generates 2–3 sentence summaries for executive dashboards, reducing reading time from minutes to seconds.

The summarization pipeline takes long text and produces a shorter version that preserves key information.

In [ ]:
summarizer = pipeline(
    "summarization",
    model=CONFIG["summarization_model"],
    device=device_id,
)
print(f"Loaded: {CONFIG['summarization_model']}")

In [ ]:
report = """
The company reported record quarterly revenue of $4.2 billion, up 18% year over year,
driven primarily by strong performance in cloud services and enterprise software licensing.
Cloud revenue alone grew 32% to $1.8 billion, reflecting continued migration of enterprise
workloads to the company's platform. Operating margins improved to 28.5% from 24.1% in the
prior year period, as the company realized economies of scale in its cloud infrastructure.
The company also announced a $2 billion share buyback program and raised its quarterly
dividend by 12%. Management noted that demand from financial services and healthcare
verticals was particularly strong, with several multi-year contracts signed during the quarter.
However, the company cautioned that macroeconomic uncertainty could impact enterprise
spending decisions in upcoming quarters.
"""

summary = summarizer(report, max_length=80, min_length=30, do_sample=False)
print("Summary:")
print(summary[0]["summary_text"])

In [ ]:
# Control output length with max_length and min_length
summary_short = summarizer(report, max_length=40, min_length=15, do_sample=False)
summary_long = summarizer(report, max_length=150, min_length=50, do_sample=False)

print("Short summary (max 40 tokens):")
print(f"  {summary_short[0]['summary_text']}\n")
print("Long summary (max 150 tokens):")
print(f"  {summary_long[0]['summary_text']}")

In [ ]:
# Batch summarization — multiple reports at once
reports = [
    "The IT security team detected 47 unauthorized access attempts over the past week, primarily targeting the customer database. All attempts were blocked by the firewall. The team recommends enabling multi-factor authentication for all admin accounts and scheduling a penetration test within the next 30 days.",
    "Customer support handled 3,200 tickets this quarter, a 15% increase from Q2. Average resolution time decreased from 4.2 hours to 3.1 hours due to the new ticket routing system. Top complaint categories were billing issues (28%), product defects (22%), and shipping delays (18%). The team hired 8 additional agents to handle growing volume.",
    "The data engineering team completed the migration of the analytics pipeline from on-premises Hadoop to cloud-based Spark. Processing time for the daily ETL job dropped from 6 hours to 45 minutes. Storage costs decreased by 40% due to columnar compression. The team will next migrate real-time streaming pipelines."
]

summaries = summarizer(reports, max_length=60, min_length=20, do_sample=False)

for i, (report_text, summ) in enumerate(zip(reports, summaries)):
    print(f"Report {i+1}: {report_text[:80]}...")
    print(f"Summary:  {summ['summary_text']}\n")

In [ ]:
# Understand input/output token counts
from transformers import AutoTokenizer

summ_tokenizer = AutoTokenizer.from_pretrained(CONFIG["summarization_model"])

input_tokens = summ_tokenizer.tokenize(report)
output_tokens = summ_tokenizer.tokenize(summary[0]["summary_text"])

print(f"Input tokens:  {len(input_tokens)}")
print(f"Output tokens: {len(output_tokens)}")
print(f"Compression:   {len(output_tokens)/len(input_tokens):.1%}")
print(f"Model max input: {summ_tokenizer.model_max_length} tokens")

### Summarization — handling long inputs

Real documents often exceed the model's maximum input length. Two strategies:
1. **Truncation**: cut the input to `max_length` tokens (simple but loses tail content)
2. **Chunking**: split the document into overlapping chunks, summarize each, then combine

In [ ]:
# Chunk-and-summarize for long documents
def chunk_text(text: str, max_tokens: int = 900, overlap: int = 100) -> list[str]:
    """Split text into token-bounded chunks with overlap."""
    tokens = summ_tokenizer.tokenize(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_str = summ_tokenizer.convert_tokens_to_string(chunk_tokens)
        chunks.append(chunk_str)
        start = end - overlap
    return chunks


# Simulate a long document by repeating the report
long_doc = (report.strip() + " ") * 5
chunks = chunk_text(long_doc)

print(f"Document tokens: {len(summ_tokenizer.tokenize(long_doc))}")
print(f"Chunks:          {len(chunks)}")

chunk_summaries = []
for i, chunk in enumerate(chunks):
    s = summarizer(chunk, max_length=60, min_length=20, do_sample=False)
    chunk_summaries.append(s[0]["summary_text"])
    print(f"  Chunk {i+1}: {s[0]['summary_text'][:80]}...")

# Combine chunk summaries into a final summary
combined = " ".join(chunk_summaries)
final = summarizer(combined, max_length=80, min_length=30, do_sample=False)
print(f"\nFinal summary: {final[0]['summary_text']}")

## 4. Question answering

**Scenario:** A compliance team maintains a set of policy documents. Instead of searching manually, an internal tool lets employees ask questions in natural language and get answers extracted directly from the relevant policy text.

The QA pipeline performs **extractive QA**: it finds the answer as a span (substring) within the provided context. It does not generate new text — it highlights the most relevant part of the input.

Day 3 extends this to **retrieval-augmented generation (RAG)**, where the system first retrieves relevant documents and then generates an answer.

In [ ]:
qa = pipeline(
    "question-answering",
    model=CONFIG["qa_model"],
    device=device_id,
)
print(f"Loaded: {CONFIG['qa_model']}")

In [ ]:
context = """
The company's data retention policy requires that all customer personal data be deleted
within 90 days of account closure, unless a legal hold is in effect. Data stored in backup
systems must be purged within 180 days. Employees who handle personal data must complete
annual privacy training. The Data Protection Officer (DPO) must be notified within 72 hours
of any suspected data breach. Third-party processors must sign a Data Processing Agreement
(DPA) before receiving any personal data. All data transfers outside the European Economic
Area require Standard Contractual Clauses or equivalent safeguards.
"""

question = "How long after account closure must personal data be deleted?"

answer = qa(question=question, context=context, truncation=True, max_length=512)
print(f"Question: {question}")
print(f"Answer:   {answer['answer']}")
print(f"Score:    {answer['score']:.4f}")
print(f"Span:     chars {answer['start']}–{answer['end']}")

In [ ]:
# Multiple questions on the same context
questions = [
    "Who must be notified of a data breach?",
    "What training must employees complete?",
    "What is required for data transfers outside the EEA?",
    "How long do backups take to purge?",
]

for q in questions:
    ans = qa(question=q, context=context, truncation=True, max_length=512)
    print(f"Q: {q}")
    print(f"A: {ans['answer']}  (score: {ans['score']:.3f})\n")

In [ ]:
# Confidence threshold — filter low-confidence answers
CONFIDENCE_THRESHOLD = 0.10

ambiguous_questions = [
    "What is the company's revenue?",  # not in context
    "When was the policy last updated?",  # not in context
    "How long for backup purge?",  # in context
]

for q in ambiguous_questions:
    ans = qa(question=q, context=context, truncation=True, max_length=512)
    status = "ACCEPT" if ans["score"] >= CONFIDENCE_THRESHOLD else "REJECT (low confidence)"
    print(f"Q: {q}")
    print(f"A: {ans['answer']}  score={ans['score']:.3f}  [{status}]\n")

In [ ]:
# QA on a different domain — technical documentation
tech_context = """
The API rate limit is 1000 requests per minute per API key. Exceeding the rate limit
returns HTTP 429 with a Retry-After header indicating the number of seconds to wait.
Authentication uses Bearer tokens passed in the Authorization header. Tokens expire
after 24 hours and must be refreshed using the /auth/refresh endpoint. The maximum
request body size is 10 MB. Responses are returned in JSON format with a top-level
'data' key for successful requests and an 'error' key for failures.
"""

tech_questions = [
    "What is the API rate limit?",
    "How long do tokens last?",
    "What is the maximum request body size?",
    "What header contains the authentication token?",
]

for q in tech_questions:
    ans = qa(question=q, context=tech_context, truncation=True, max_length=512)
    print(f"Q: {q}")
    print(f"A: {ans['answer']}  (score: {ans['score']:.3f})\n")

### Multi-document QA pattern

When the context is spread across multiple documents, run QA against each document and pick the answer with the highest confidence score.

In [ ]:
# Multi-document QA: search across multiple policy sections
documents = {
    "Security Policy": "All employees must use multi-factor authentication (MFA) for VPN access. Passwords must be at least 14 characters and rotated every 90 days. USB storage devices are prohibited on company networks.",
    "Leave Policy": "Employees accrue 20 days of paid time off per year. Unused PTO carries over up to a maximum of 10 days. Parental leave is 16 weeks for primary caregivers and 6 weeks for secondary caregivers.",
    "Expense Policy": "Business travel expenses must be submitted within 30 days of travel. Flights over $500 require manager approval before booking. Per diem rates follow GSA guidelines for the destination city.",
}

question_multi = "How many days of paid time off do employees get?"

best_answer = None
best_score = -1.0
best_doc = ""

for doc_name, doc_text in documents.items():
    ans = qa(question=question_multi, context=doc_text, truncation=True, max_length=512)
    print(f"  {doc_name:20s}  answer='{ans['answer']}'  score={ans['score']:.4f}")
    if ans["score"] > best_score:
        best_score = ans["score"]
        best_answer = ans["answer"]
        best_doc = doc_name

print(f"\nBest answer: '{best_answer}' from '{best_doc}' (score: {best_score:.4f})")

## 5. Sentiment analysis

**Scenario:** A customer experience team processes product reviews and support tickets. Sentiment analysis classifies each text as positive or negative, enabling automatic routing of negative feedback to escalation queues and positive feedback to testimonial pipelines.

In [ ]:
sentiment = pipeline(
    "sentiment-analysis",
    model=CONFIG["sentiment_model"],
    device=device_id,
)
print(f"Loaded: {CONFIG['sentiment_model']}")

In [ ]:
# Single text
result = sentiment("The new dashboard is intuitive and makes reporting much faster.", truncation=True, max_length=512)
print(result)

In [ ]:
# Batch: classify multiple support tickets
tickets = [
    "The software crashes every time I try to export a PDF. This has been happening for 3 days.",
    "Your customer support team resolved my issue within 20 minutes. Excellent experience.",
    "I cannot log in after the latest update. The password reset email never arrives.",
    "The onboarding process was smooth and the documentation is clear and helpful.",
    "We have been waiting 2 weeks for a response to our enterprise license inquiry.",
    "The API integration was straightforward — connected our systems in under an hour.",
    "The mobile app drains battery excessively and heats up the phone during use.",
    "Great value for the price. The features exceed what we expected from this tier.",
]

results = sentiment(tickets, truncation=True, max_length=512)

for ticket, res in zip(tickets, results):
    print(f"  [{res['label']:>8s} {res['score']:.3f}]  {ticket[:75]}")

In [ ]:
# Route tickets based on sentiment — production pattern
def route_ticket(text: str, result: dict) -> str:
    """Route a ticket based on sentiment and confidence."""
    if result["label"] == "NEGATIVE" and result["score"] > 0.95:
        return "ESCALATE"
    elif result["label"] == "NEGATIVE":
        return "REVIEW"
    elif result["label"] == "POSITIVE" and result["score"] > 0.95:
        return "TESTIMONIAL"
    else:
        return "STANDARD"


for ticket, res in zip(tickets, results):
    route = route_ticket(ticket, res)
    print(f"  [{route:>12s}]  {ticket[:65]}")

In [ ]:
# Aggregate sentiment statistics
from collections import Counter

label_counts = Counter(r["label"] for r in results)
avg_score = sum(r["score"] for r in results) / len(results)

print("Sentiment distribution:")
for label, count in label_counts.items():
    print(f"  {label}: {count} ({count/len(results):.0%})")
print(f"\nAverage confidence: {avg_score:.3f}")

In [ ]:
# Sentiment on multi-sentence text — the model scores the entire input
long_review = (
    "The initial setup was painfully slow and the documentation was confusing. "
    "However, once configured, the system works reliably and the performance is impressive. "
    "Support was responsive when we finally reached them. Overall a mixed experience."
)

result_long = sentiment(long_review, truncation=True, max_length=512)
print(f"Text: {long_review[:80]}...")
print(f"Overall sentiment: {result_long[0]['label']} ({result_long[0]['score']:.3f})")
print("\nNote: the model gives one label for the entire text, not per-sentence.")

In [ ]:
# Per-sentence sentiment breakdown
sentences = [s.strip() for s in long_review.split(".") if s.strip()]
sentence_results = sentiment(sentences, truncation=True, max_length=512)

print("Per-sentence breakdown:\n")
for sent, res in zip(sentences, sentence_results):
    print(f"  [{res['label']:>8s} {res['score']:.3f}]  {sent}")

In [ ]:
# Map model labels to business labels
LABEL_MAP = {
    "POSITIVE": "satisfied",
    "NEGATIVE": "dissatisfied",
}

for ticket, res in zip(tickets[:4], results[:4]):
    business_label = LABEL_MAP.get(res["label"], "unknown")
    print(f"  [{business_label:>13s} {res['score']:.3f}]  {ticket[:65]}")

## 6. Image classification

**Scenario:** A document processing pipeline classifies incoming scanned images by type (invoice, receipt, letter, form) to route them to the correct processing queue. Or a content moderation system classifies uploaded images.

The image classification pipeline takes an image (file path, URL, or PIL Image) and returns top-k predicted labels with confidence scores.

In [ ]:
image_classifier = pipeline(
    "image-classification",
    model=CONFIG["image_classification_model"],
    device=device_id,
)
print(f"Loaded: {CONFIG['image_classification_model']}")

In [ ]:
# Classify an image from a URL (publicly available test image)
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat.png"

results_img = image_classifier(image_url)

print(f"Image: {image_url.split('/')[-1]}\n")
print("Top predictions:")
for r in results_img[:5]:
    bar = "#" * int(r["score"] * 40)
    print(f"  {r['label']:30s}  {r['score']:.4f}  {bar}")

In [ ]:
# Classify from a PIL Image object
from PIL import Image
import requests
from io import BytesIO

response = requests.get(image_url, timeout=30)
img = Image.open(BytesIO(response.content))

results_pil = image_classifier(img)

print(f"Image size: {img.size}")
print(f"Top label:  {results_pil[0]['label']} ({results_pil[0]['score']:.4f})")

In [ ]:
# Second image — different category
image_url_2 = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png"

results_img_2 = image_classifier(image_url_2)

print(f"Image: {image_url_2.split('/')[-1]}\n")
print("Top predictions:")
for r in results_img_2[:5]:
    bar = "#" * int(r["score"] * 40)
    print(f"  {r['label']:30s}  {r['score']:.4f}  {bar}")

In [ ]:
# Confidence filtering for production — only accept high-confidence predictions
CONFIDENCE_THRESHOLD_IMG = 0.30

test_images = [image_url, image_url_2]

for url in test_images:
    preds = image_classifier(url)
    top = preds[0]
    status = "ACCEPT" if top["score"] >= CONFIDENCE_THRESHOLD_IMG else "MANUAL REVIEW"
    print(f"Image: {url.split('/')[-1]:30s}  {top['label']:25s}  {top['score']:.3f}  [{status}]")

In [ ]:
# Batch image processing — load multiple images
urls = [image_url, image_url_2]
images = []
for u in urls:
    resp = requests.get(u, timeout=30)
    images.append(Image.open(BytesIO(resp.content)))

# Classify batch
batch_results = image_classifier(images)

for u, res in zip(urls, batch_results):
    top = res[0] if isinstance(res, list) else res
    print(f"  {u.split('/')[-1]:35s}  {top['label']:25s}  {top['score']:.3f}")

## 8. Frameworks used in this notebook

| Function / Class | Library | Purpose | Example |
|-----------------|---------|---------|---------|
| `pipeline()` | transformers | High-level task inference | `pipeline("summarization", model="...")` |
| `AutoTokenizer.from_pretrained()` | transformers | Load tokenizer for a model | `AutoTokenizer.from_pretrained("bert-base")` |
| `AutoModel.from_pretrained()` | transformers | Load model weights | `AutoModel.from_pretrained("bert-base")` |
| `Image.open()` | Pillow | Load/manipulate images | `Image.open(BytesIO(response.content))` |

Module 01 has the full frameworks reference for all Day 2 libraries.